
# RandLA-Net Training + Prediction (NPM3D mini)

This notebook builds a practical **RandLA-Net style** point-cloud segmentation pipeline using PyTorch:
- train on `training/*.ply`
- optional validation on one held-out scene
- fine-tune on **all** training scenes
- predict `test/MiniDijon9.ply`
- export `submission_randlanet.txt`

If you have GPU/CUDA, training will use it automatically. On Apple Silicon, it will use MPS when available.


In [ ]:

# Install missing dependencies (run once)
import sys
import subprocess
import importlib.util


def ensure_module(import_name: str, pip_args: list[str]):
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {import_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", *pip_args])


# For CUDA users, you may prefer the official CUDA wheel index:
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
ensure_module("torch", ["torch", "torchvision", "torchaudio"])
ensure_module("plyfile", ["plyfile"])
ensure_module("tqdm", ["tqdm"])
print("Dependencies ready.")


In [ ]:

import os
import glob
import math
import random
from dataclasses import dataclass

import numpy as np
from tqdm.auto import tqdm
from plyfile import PlyData
from scipy.spatial import cKDTree

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# -------------------------
# Reproducibility + device
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Using device:", DEVICE)


# -------------------------
# Paths and config
# -------------------------
TRAIN_GLOB = "training/*.ply"
TEST_PLY = "test/MiniDijon9.ply"
LABEL_FIELD = "class"
NUM_CLASSES = 6
IGNORE_LABEL = 255

OUTPUT_CKPT = "randlanet_best.pt"
OUTPUT_SUBMISSION = "submission_randlanet.txt"

@dataclass
class CFG:
    # Data
    num_points: int = 2048
    tile_size: float = 10.0
    batch_size: int = 6
    num_workers: int = 0

    # Model
    k_neighbors: int = 16

    # Training schedule
    epochs_stage1: int = 24
    epochs_stage2: int = 8
    lr_stage1: float = 1e-3
    lr_stage2: float = 3e-4
    weight_decay: float = 1e-4

    # Validation
    val_scene_name: str = "MiniParis1.ply"  # hold out one scene for tracking

cfg = CFG()
print(cfg)


In [ ]:

# -------------------------
# IO + preprocessing utils
# -------------------------
def load_xyz_labels(path: str, label_field: str | None = None):
    ply = PlyData.read(path)
    v = ply["vertex"].data
    xyz = np.vstack([v["x"], v["y"], v["z"]]).T.astype(np.float32)
    labels = None
    if label_field is not None:
        if label_field not in v.dtype.names:
            raise ValueError(f"Label field '{label_field}' not in {path}. Found: {v.dtype.names}")
        labels = np.asarray(v[label_field], dtype=np.int32)
    return xyz, labels, v.dtype.names


def make_scene_dict(path: str, with_labels: bool):
    xyz, labels, fields = load_xyz_labels(path, LABEL_FIELD if with_labels else None)
    out = {
        "name": os.path.basename(path),
        "path": path,
        "xyz": xyz,
        "min": xyz.min(axis=0).astype(np.float32),
        "max": xyz.max(axis=0).astype(np.float32),
        "fields": fields,
    }
    if with_labels:
        out["labels"] = labels
    return out


def build_tile_chunks(xyz: np.ndarray, num_points: int, tile_size: float, seed: int = 42):
    """
    Partition points in XY tiles, then split each tile into fixed-size chunks.
    This covers all points each epoch (padding only for the last chunk per tile).
    """
    rng = np.random.default_rng(seed)
    x = xyz[:, 0]
    y = xyz[:, 1]

    x0 = float(x.min())
    y0 = float(y.min())
    ix = np.floor((x - x0) / tile_size).astype(np.int64)
    iy = np.floor((y - y0) / tile_size).astype(np.int64)
    key = (ix << 32) | (iy & 0xFFFFFFFF)

    order = np.argsort(key)
    key_sorted = key[order]
    split_pts = np.flatnonzero(np.diff(key_sorted)) + 1
    groups = np.split(order, split_pts)

    chunks = []
    for g in groups:
        g = np.asarray(g, dtype=np.int64)
        if len(g) == 0:
            continue
        rng.shuffle(g)
        for s in range(0, len(g), num_points):
            chunk = g[s:s + num_points]
            if len(chunk) < num_points:
                pad = rng.choice(g, size=(num_points - len(chunk)), replace=True)
                chunk = np.concatenate([chunk, pad])
            chunks.append(chunk.astype(np.int64, copy=False))

    return chunks


def make_input_features(xyz_block: np.ndarray, scene_min: np.ndarray, scene_max: np.ndarray):
    # Local coordinates around block centroid
    center = xyz_block.mean(axis=0, keepdims=True)
    local = xyz_block - center

    # Scene-normalized absolute coordinates
    span = np.maximum(scene_max - scene_min, 1e-6)
    norm = (xyz_block - scene_min) / span

    feats = np.concatenate([local, norm], axis=1).astype(np.float32)
    coords = local.astype(np.float32)
    return coords, feats


In [ ]:

# -------------------------
# Datasets
# -------------------------
class ChunkDataset(Dataset):
    def __init__(self, scenes, chunks_per_scene, augment: bool):
        self.scenes = scenes
        self.samples = []  # (scene_idx, chunk_indices)
        self.augment = augment

        for si, chunks in enumerate(chunks_per_scene):
            for c in chunks:
                self.samples.append((si, c))

    def __len__(self):
        return len(self.samples)

    def _augment(self, coords: np.ndarray, feats: np.ndarray):
        # Random Z rotation
        theta = np.random.uniform(0.0, 2.0 * np.pi)
        c = np.cos(theta)
        s = np.sin(theta)
        R = np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
        coords = coords @ R.T

        # Scale + jitter
        scale = np.random.uniform(0.95, 1.05)
        coords = coords * scale
        coords = coords + np.random.normal(0.0, 0.01, size=coords.shape).astype(np.float32)

        feats = feats.copy()
        feats[:, :3] = coords
        return coords, feats

    def __getitem__(self, idx):
        scene_idx, point_idx = self.samples[idx]
        scene = self.scenes[scene_idx]

        xyz = scene["xyz"][point_idx]
        labels_raw = scene["labels"][point_idx]

        coords, feats = make_input_features(xyz, scene["min"], scene["max"])
        if self.augment:
            coords, feats = self._augment(coords, feats)

        labels = labels_raw.astype(np.int64)
        # map 1..6 -> 0..5 ; ignore 0
        labels = np.where(labels == 0, IGNORE_LABEL, labels - 1).astype(np.int64)

        return (
            torch.from_numpy(coords),
            torch.from_numpy(feats),
            torch.from_numpy(labels),
        )


class TestChunkDataset(Dataset):
    def __init__(self, scene, chunks):
        self.scene = scene
        self.chunks = chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        point_idx = self.chunks[idx]
        xyz = self.scene["xyz"][point_idx]
        coords, feats = make_input_features(xyz, self.scene["min"], self.scene["max"])
        return torch.from_numpy(coords), torch.from_numpy(feats), torch.from_numpy(point_idx.astype(np.int64))


In [ ]:

# -------------------------
# RandLA-Net style model
# -------------------------
class SharedMLP1D(nn.Module):
    def __init__(self, in_c, out_c, act=True):
        super().__init__()
        self.conv = nn.Conv1d(in_c, out_c, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm1d(out_c)
        self.act = nn.LeakyReLU(0.2, inplace=True) if act else nn.Identity()

    def forward(self, x):
        x = x.contiguous()
        return self.act(self.bn(self.conv(x)))


class SharedMLP2D(nn.Module):
    def __init__(self, in_c, out_c, act=True):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_c)
        self.act = nn.LeakyReLU(0.2, inplace=True) if act else nn.Identity()

    def forward(self, x):
        x = x.contiguous()
        return self.act(self.bn(self.conv(x)))


def knn_indices(coords: torch.Tensor, k: int):
    # coords: [B, N, 3]
    d = torch.cdist(coords, coords)  # [B, N, N]
    idx = d.topk(k=k + 1, dim=-1, largest=False).indices[:, :, 1:]
    return idx


def gather_points(points: torch.Tensor, idx: torch.Tensor):
    # points: [B, N, C]
    # idx: [B, M] or [B, M, K]
    B = points.shape[0]
    if idx.dim() == 2:
        b = torch.arange(B, device=points.device)[:, None]
        return points[b, idx]
    if idx.dim() == 3:
        b = torch.arange(B, device=points.device)[:, None, None]
        return points[b, idx]
    raise ValueError("idx must have rank 2 or 3")


def sample_points(coords: torch.Tensor, feats: torch.Tensor, ratio: int, training: bool):
    B, N, _ = coords.shape
    M = max(N // ratio, 16)
    idx_list = []
    for _ in range(B):
        if training:
            idx = torch.randperm(N, device=coords.device)[:M]
        else:
            idx = torch.linspace(0, N - 1, M, device=coords.device).long()
        idx_list.append(idx)
    idx = torch.stack(idx_list, dim=0)

    coords_s = gather_points(coords, idx)
    feats_s = gather_points(feats, idx)
    return coords_s, feats_s


def nearest_interpolate(src_coords: torch.Tensor, src_feat: torch.Tensor, tgt_coords: torch.Tensor):
    # src: coarse, tgt: fine
    d = torch.cdist(tgt_coords, src_coords)  # [B, N_tgt, N_src]
    idx = d.argmin(dim=-1)  # [B, N_tgt]
    return gather_points(src_feat, idx)


class LocalFeatureAggregation(nn.Module):
    def __init__(self, in_c: int, out_c: int, k: int):
        super().__init__()
        self.k = k
        geo_c = max(8, out_c // 4)
        mid_c = out_c

        self.geo_mlp = SharedMLP2D(4, geo_c, act=True)
        self.feat_mlp = SharedMLP2D((2 * in_c) + geo_c, mid_c, act=True)
        self.attn_mlp = nn.Conv2d(mid_c, 1, kernel_size=1, bias=True)
        self.out_mlp = SharedMLP1D(mid_c, out_c, act=True)

    def forward(self, coords: torch.Tensor, feats: torch.Tensor):
        # coords: [B,N,3], feats: [B,N,C]
        B, N, C = feats.shape
        idx = knn_indices(coords.contiguous(), self.k)  # [B,N,k]

        neigh_feats = gather_points(feats.contiguous(), idx)      # [B,N,k,C]
        center_feats = feats.unsqueeze(2).expand(-1, -1, self.k, -1)

        neigh_xyz = gather_points(coords.contiguous(), idx)       # [B,N,k,3]
        rel_xyz = neigh_xyz - coords.unsqueeze(2)    # [B,N,k,3]
        rel_dist = torch.norm(rel_xyz, dim=-1, keepdim=True)

        geo = torch.cat([rel_xyz, rel_dist], dim=-1)  # [B,N,k,4]
        geo = self.geo_mlp(geo.permute(0, 3, 1, 2).contiguous()).permute(0, 2, 3, 1).contiguous()

        f = torch.cat([center_feats, neigh_feats - center_feats, geo], dim=-1)
        f = self.feat_mlp(f.permute(0, 3, 1, 2).contiguous())

        attn = torch.softmax(self.attn_mlp(f), dim=-1)
        f = (f * attn).sum(dim=-1)  # [B,C,N]

        out = self.out_mlp(f).permute(0, 2, 1).contiguous()
        return out


class RandLABlock(nn.Module):
    def __init__(self, in_c: int, out_c: int, k: int):
        super().__init__()
        inter_c = max(16, out_c // 2)
        self.pre = SharedMLP1D(in_c, inter_c, act=True)
        self.lfa = LocalFeatureAggregation(inter_c, out_c, k=k)
        self.post = SharedMLP1D(out_c, out_c, act=False)
        self.short = SharedMLP1D(in_c, out_c, act=False)

    def forward(self, coords: torch.Tensor, feats: torch.Tensor):
        x = self.pre(feats.permute(0, 2, 1).contiguous()).permute(0, 2, 1).contiguous()
        x = self.lfa(coords, x)
        x = self.post(x.permute(0, 2, 1).contiguous()).permute(0, 2, 1).contiguous()
        s = self.short(feats.permute(0, 2, 1).contiguous()).permute(0, 2, 1).contiguous()
        return F.leaky_relu(x + s, negative_slope=0.2)


class RandLANetSmall(nn.Module):
    def __init__(self, num_classes: int = 6, k_neighbors: int = 16):
        super().__init__()
        self.stem = SharedMLP1D(6, 16, act=True)

        self.enc1 = RandLABlock(16, 64, k_neighbors)
        self.enc2 = RandLABlock(64, 128, k_neighbors)
        self.bottleneck = RandLABlock(128, 256, k_neighbors)

        self.dec2 = SharedMLP1D(256 + 128, 128, act=True)
        self.dec1 = SharedMLP1D(128 + 64, 96, act=True)

        self.head1 = SharedMLP1D(96, 64, act=True)
        self.drop = nn.Dropout(0.3)
        self.head2 = nn.Conv1d(64, num_classes, kernel_size=1)

    def forward(self, coords: torch.Tensor, feats: torch.Tensor):
        # coords: [B,N,3], feats: [B,N,6]
        x0 = self.stem(feats.permute(0, 2, 1).contiguous()).permute(0, 2, 1).contiguous()

        f1 = self.enc1(coords, x0)
        c2, f2_in = sample_points(coords, f1, ratio=4, training=self.training)

        f2 = self.enc2(c2, f2_in)
        c3, f3_in = sample_points(c2, f2, ratio=4, training=self.training)

        f3 = self.bottleneck(c3, f3_in)

        u2 = nearest_interpolate(c3, f3, c2)
        u2 = self.dec2(torch.cat([u2, f2], dim=-1).permute(0, 2, 1).contiguous()).permute(0, 2, 1).contiguous()

        u1 = nearest_interpolate(c2, u2, coords)
        u1 = self.dec1(torch.cat([u1, f1], dim=-1).permute(0, 2, 1).contiguous()).permute(0, 2, 1).contiguous()

        out = self.head1(u1.permute(0, 2, 1).contiguous())
        out = self.drop(out)
        logits = self.head2(out).permute(0, 2, 1).contiguous()
        return logits


In [ ]:

# -------------------------
# Training / metrics
# -------------------------
def compute_miou_from_cm(cm: np.ndarray):
    ious = []
    for c in range(cm.shape[0]):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        denom = tp + fp + fn
        ious.append(0.0 if denom == 0 else tp / denom)
    return np.array(ious, dtype=np.float64), float(np.mean(ious))


def run_epoch(model, loader, optimizer, class_weights=None, train=True):
    if train:
        model.train()
    else:
        model.eval()

    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    total_loss = 0.0
    n_batches = 0

    criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=IGNORE_LABEL)

    use_amp = (DEVICE.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    for coords, feats, labels in tqdm(loader, leave=False):
        coords = coords.to(DEVICE, non_blocking=True)
        feats = feats.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(coords, feats)  # [B,N,C]
                # CE expects [B,C,N] with target [B,N]; keep tensors contiguous to avoid stride/view issues.
                loss = criterion(logits.permute(0, 2, 1).contiguous(), labels.contiguous())

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += float(loss.item())
        n_batches += 1

        with torch.no_grad():
            pred = logits.argmax(dim=-1)
            valid = labels != IGNORE_LABEL
            t = labels[valid].detach().cpu().numpy().astype(np.int64)
            p = pred[valid].detach().cpu().numpy().astype(np.int64)
            np.add.at(cm, (t, p), 1)

    ious, miou = compute_miou_from_cm(cm)
    mean_loss = total_loss / max(1, n_batches)
    return mean_loss, miou, ious, cm


In [ ]:

# -------------------------
# Load scenes + build datasets
# -------------------------
train_paths = sorted(glob.glob(TRAIN_GLOB))
assert len(train_paths) > 0, f"No training files found for {TRAIN_GLOB}"
assert os.path.exists(TEST_PLY), f"Missing test file: {TEST_PLY}"

all_scenes = [make_scene_dict(p, with_labels=True) for p in train_paths]
print("Loaded training scenes:")
for s in all_scenes:
    counts = np.bincount(np.clip(s["labels"], 0, NUM_CLASSES), minlength=NUM_CLASSES + 1)
    print(f" - {s['name']}: N={len(s['xyz'])}, labels[0..6]={counts.tolist()}")

val_scenes = [s for s in all_scenes if s["name"] == cfg.val_scene_name]
train_scenes = [s for s in all_scenes if s["name"] != cfg.val_scene_name]

if len(val_scenes) == 0:
    # fallback if configured name not present
    val_scenes = [all_scenes[-1]]
    train_scenes = all_scenes[:-1]

print("\nStage-1 split:")
print(" Train scenes:", [s["name"] for s in train_scenes])
print(" Val scenes  :", [s["name"] for s in val_scenes])


def build_chunks_for_scenes(scenes, num_points, tile_size, seed_offset=0):
    chunks_per_scene = []
    for i, s in enumerate(scenes):
        chunks = build_tile_chunks(s["xyz"], num_points=num_points, tile_size=tile_size, seed=SEED + seed_offset + i)
        chunks_per_scene.append(chunks)
        print(f"{s['name']}: {len(chunks)} chunks")
    return chunks_per_scene


print("\nBuilding train/val chunks...")
train_chunks = build_chunks_for_scenes(train_scenes, cfg.num_points, cfg.tile_size, seed_offset=100)
val_chunks = build_chunks_for_scenes(val_scenes, cfg.num_points, cfg.tile_size, seed_offset=200)

train_ds = ChunkDataset(train_scenes, train_chunks, augment=True)
val_ds = ChunkDataset(val_scenes, val_chunks, augment=False)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=False,
)

# class weights from stage-1 train set
all_train_labels = np.concatenate([s["labels"] for s in train_scenes], axis=0)
all_train_labels = all_train_labels[all_train_labels > 0] - 1
counts = np.bincount(all_train_labels, minlength=NUM_CLASSES).astype(np.float64)
weights = counts.sum() / np.maximum(counts, 1.0)
weights = weights / weights.mean()
class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
print("Class weights:", class_weights.detach().cpu().numpy())

print(f"Dataset sizes: train={len(train_ds)} chunks, val={len(val_ds)} chunks")


In [ ]:

# -------------------------
# Stage 1: train with validation tracking
# -------------------------
model = RandLANetSmall(num_classes=NUM_CLASSES, k_neighbors=cfg.k_neighbors).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr_stage1, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs_stage1)

best_miou = -1.0
history = []

for epoch in range(1, cfg.epochs_stage1 + 1):
    tr_loss, tr_miou, tr_ious, _ = run_epoch(model, train_loader, optimizer, class_weights=class_weights, train=True)
    va_loss, va_miou, va_ious, _ = run_epoch(model, val_loader, optimizer, class_weights=class_weights, train=False)
    scheduler.step()

    history.append((epoch, tr_loss, tr_miou, va_loss, va_miou))
    print(
        f"Epoch {epoch:02d}/{cfg.epochs_stage1} | "
        f"train loss={tr_loss:.4f}, mIoU={tr_miou:.4f} | "
        f"val loss={va_loss:.4f}, mIoU={va_miou:.4f}"
    )

    if va_miou > best_miou:
        best_miou = va_miou
        torch.save(
            {
                "model_state": model.state_dict(),
                "best_val_miou": best_miou,
                "cfg": cfg.__dict__,
            },
            OUTPUT_CKPT,
        )
        print(f"  -> New best checkpoint saved: {OUTPUT_CKPT}")

print("Best stage-1 val mIoU:", best_miou)


In [ ]:

# -------------------------
# Stage 2: fine-tune on ALL training scenes (best for leaderboard)
# -------------------------
ckpt = torch.load(OUTPUT_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])

all_chunks = build_chunks_for_scenes(all_scenes, cfg.num_points, cfg.tile_size, seed_offset=300)
all_train_ds = ChunkDataset(all_scenes, all_chunks, augment=True)
all_train_loader = DataLoader(
    all_train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=True,
)

# recompute class weights on all data
labels_all = np.concatenate([s["labels"] for s in all_scenes], axis=0)
labels_all = labels_all[labels_all > 0] - 1
counts_all = np.bincount(labels_all, minlength=NUM_CLASSES).astype(np.float64)
weights_all = counts_all.sum() / np.maximum(counts_all, 1.0)
weights_all = weights_all / weights_all.mean()
class_weights_all = torch.tensor(weights_all, dtype=torch.float32, device=DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr_stage2, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, cfg.epochs_stage2))

for epoch in range(1, cfg.epochs_stage2 + 1):
    tr_loss, tr_miou, tr_ious, _ = run_epoch(
        model,
        all_train_loader,
        optimizer,
        class_weights=class_weights_all,
        train=True,
    )
    scheduler.step()
    print(f"Fine-tune {epoch:02d}/{cfg.epochs_stage2} | loss={tr_loss:.4f}, mIoU={tr_miou:.4f}")

torch.save({"model_state": model.state_dict(), "cfg": cfg.__dict__}, OUTPUT_CKPT)
print("Final model saved to", OUTPUT_CKPT)


In [ ]:

# -------------------------
# Inference on test cloud + submission export
# -------------------------
@torch.no_grad()
def predict_test_scene(model, scene, num_points, tile_size, batch_size, num_workers=0):
    model.eval()

    chunks = build_tile_chunks(scene["xyz"], num_points=num_points, tile_size=tile_size, seed=SEED + 999)
    print(f"Test chunks: {len(chunks)}")
    test_ds = TestChunkDataset(scene, chunks)
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )

    n = len(scene["xyz"])
    proba_sum = np.zeros((n, NUM_CLASSES), dtype=np.float32)
    count = np.zeros(n, dtype=np.float32)

    for coords, feats, point_idx in tqdm(test_loader):
        coords = coords.to(DEVICE, non_blocking=True)
        feats = feats.to(DEVICE, non_blocking=True)

        logits = model(coords, feats)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()  # [B,N,C]
        idx_np = point_idx.numpy()  # [B,N]

        for b in range(idx_np.shape[0]):
            ids = idx_np[b]
            proba_sum[ids] += probs[b]
            count[ids] += 1.0

    unseen = np.where(count == 0)[0]
    if len(unseen) > 0:
        print(f"Warning: {len(unseen)} unseen points, filling with nearest seen point votes.")
        seen = np.where(count > 0)[0]
        tree = cKDTree(scene["xyz"][seen, :2])
        _, nn = tree.query(scene["xyz"][unseen, :2], k=1)
        proba_sum[unseen] = proba_sum[seen[nn]]
        count[unseen] = 1.0

    proba = proba_sum / count[:, None]
    pred_0_5 = np.argmax(proba, axis=1).astype(np.int32)
    pred_1_6 = pred_0_5 + 1
    return pred_1_6


# Load test scene
test_scene = make_scene_dict(TEST_PLY, with_labels=False)
print("Test scene:", test_scene["name"], "N=", len(test_scene["xyz"]))

# Predict
pred_1_6 = predict_test_scene(
    model,
    test_scene,
    num_points=cfg.num_points,
    tile_size=cfg.tile_size,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
)

np.savetxt(OUTPUT_SUBMISSION, pred_1_6, fmt="%d")
print(f"Saved {OUTPUT_SUBMISSION} with {len(pred_1_6)} labels in [1..6].")



## Notes for stronger results

- Increase `cfg.num_points` to `4096` if GPU memory allows.
- Increase `cfg.epochs_stage1` / `cfg.epochs_stage2` for better convergence.
- Use smaller `cfg.tile_size` (e.g. `8.0`) to improve local context consistency.
- Add test-time augmentation (multiple rotations) and average probabilities.
